<a href="https://colab.research.google.com/github/sanadv/CS360/blob/main/Maximum_flow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import deque, defaultdict


def build_residual(capacity):
    # residual[u][v] = spare capacity from u to v
    residual = defaultdict(lambda: defaultdict(int))
    for u in capacity:
        for v, c in capacity[u].items():
            residual[u][v] += c      # forward edge: full capacity
            residual[v][u] += 0      # backward edge: no flow to undo yet
    print(residual)
    return residual


def edmonds_karp(capacity, s, t):
    residual = build_residual(capacity)
    max_flow = 0
    while True:
        # 1. BFS for the shortest augmenting path
        parent = {s: None}
        queue = deque([s])
        while queue and t not in parent:
            u = queue.popleft()
            for v, cap in residual[u].items():
                if cap > 0 and v not in parent:
                    parent[v] = u
                    queue.append(v)
        if t not in parent:          # no augmenting path: done
            return max_flow, residual

        # 2. Bottleneck = smallest residual on the path
        bottleneck = float('inf')
        v = t
        while v != s:
            u = parent[v]
            bottleneck = min(bottleneck, residual[u][v])
            v = u

        # 3. Augment: forward loses, backward gains
        v = t
        while v != s:
            u = parent[v]
            residual[u][v] -= bottleneck
            residual[v][u] += bottleneck
            v = u
        max_flow += bottleneck


def ford_fulkerson(capacity, s, t):
    residual = build_residual(capacity)

    def dfs(u, visited, pushed):
        if u == t:
            return pushed
        visited.add(u)
        for v, cap in residual[u].items():
            if cap > 0 and v not in visited:
                f = dfs(v, visited, min(pushed, cap))
                if f > 0:
                    residual[u][v] -= f
                    residual[v][u] += f
                    return f
        return 0

    max_flow = 0
    while True:
        f = dfs(s, set(), float('inf'))
        if f == 0:
            return max_flow, residual
        max_flow += f


def min_cut(capacity, residual, s):
    # S = vertices still reachable from s in the final residual graph
    S, stack = {s}, [s]
    while stack:
        u = stack.pop()
        for v, cap in residual[u].items():
            if cap > 0 and v not in S:
                S.add(v)
                stack.append(v)
    cut = [(u, v, c) for u in capacity for v, c in capacity[u].items()
           if u in S and v not in S]
    return S, cut


if __name__ == "__main__":
    network = {
        's': {'A': 10, 'C': 10},
        'A': {'B': 4, 'C': 2, 'D': 8},
        'C': {'D': 9},
        'D': {'B': 6, 't': 10},
        'B': {'t': 10},
    }

    flow, residual = edmonds_karp(network, 's', 't')
    print("Edmonds-Karp max flow:", flow)

    for u in network:
        for v, c in network[u].items():
            print(f"  {u} -> {v}: {c - residual[u][v]}/{c}")

    S, cut = min_cut(network, residual, 's')
    print("Min cut S =", sorted(S), "edges:", cut)

    flow, _ = ford_fulkerson(network, 's', 't')
    print("Ford-Fulkerson (DFS) max flow:", flow)

defaultdict(<function build_residual.<locals>.<lambda> at 0x7fd422159940>, {'s': defaultdict(<class 'int'>, {'A': 10, 'C': 10}), 'A': defaultdict(<class 'int'>, {'s': 0, 'B': 4, 'C': 2, 'D': 8}), 'C': defaultdict(<class 'int'>, {'s': 0, 'A': 0, 'D': 9}), 'B': defaultdict(<class 'int'>, {'A': 0, 'D': 0, 't': 10}), 'D': defaultdict(<class 'int'>, {'A': 0, 'C': 0, 'B': 6, 't': 10}), 't': defaultdict(<class 'int'>, {'D': 0, 'B': 0})})
Edmonds-Karp max flow: 19
  s -> A: 10/10
  s -> C: 9/10
  A -> B: 4/4
  A -> C: 0/2
  A -> D: 6/8
  C -> D: 9/9
  D -> B: 5/6
  D -> t: 10/10
  B -> t: 9/10
Min cut S = ['C', 's'] edges: [('s', 'A', 10), ('C', 'D', 9)]
defaultdict(<function build_residual.<locals>.<lambda> at 0x7fd42215ae80>, {'s': defaultdict(<class 'int'>, {'A': 10, 'C': 10}), 'A': defaultdict(<class 'int'>, {'s': 0, 'B': 4, 'C': 2, 'D': 8}), 'C': defaultdict(<class 'int'>, {'s': 0, 'A': 0, 'D': 9}), 'B': defaultdict(<class 'int'>, {'A': 0, 'D': 0, 't': 10}), 'D': defaultdict(<class 'int'>